In [2]:
import requests
import pandas as pd
import datetime


In [6]:
import requests
import pandas as pd
import time

API_KEY = "aa6877ac64bbbc776b89c98b61b11b54"
lat = 24.8607
lon = 67.0011

# Define start and end dates
start_date = pd.to_datetime("2025-01-22")
end_date = pd.to_datetime("2026-01-22")

# Convert to timestamps
start_ts = int(start_date.timestamp())
end_ts = int(end_date.timestamp())

# OpenWeatherMap allows limited range per request, so chunk by 30 days
chunk_days = 30
chunk_seconds = chunk_days * 24 * 60 * 60

all_data = []

current_start = start_ts
while current_start < end_ts:
    current_end = min(current_start + chunk_seconds, end_ts)
    
    params = {
        'lat': lat,
        'lon': lon,
        'start': current_start,
        'end': current_end,
        'appid': API_KEY
    }
    
    response = requests.get("http://api.openweathermap.org/data/2.5/air_pollution/history", params=params)
    response.raise_for_status()
    data = response.json()
    
    if 'list' in data:
        all_data.extend(data['list'])
    
    print(f"Fetched data from {current_start} to {current_end}, total records: {len(data.get('list', []))}")
    current_start = current_end + 1
    time.sleep(1)  # avoid hitting rate limits

# Convert to DataFrame



Fetched data from 1737504000 to 1740096000, total records: 721
Fetched data from 1740096001 to 1742688001, total records: 672
Fetched data from 1742688002 to 1745280002, total records: 528
Fetched data from 1745280003 to 1747872003, total records: 720
Fetched data from 1747872004 to 1750464004, total records: 720
Fetched data from 1750464005 to 1753056005, total records: 720
Fetched data from 1753056006 to 1755648006, total records: 720
Fetched data from 1755648007 to 1758240007, total records: 720
Fetched data from 1758240008 to 1760832008, total records: 720
Fetched data from 1760832009 to 1763424009, total records: 720
Fetched data from 1763424010 to 1766016010, total records: 720
Fetched data from 1766016011 to 1768608011, total records: 696
Fetched data from 1768608012 to 1769040000, total records: 96


In [26]:
raw_df = pd.DataFrame(all_data)
raw_df['datetime'] = pd.to_datetime(raw_df['dt'], unit='s')
raw_df.head()

,main,components,dt,datetime
0,{'aqi': 4},"{'co': 1201.63, 'no': 0, 'no2': 28.45, 'o3': 7...",1737504000,2025-01-22 00:00:00
1,{'aqi': 5},"{'co': 1508.71, 'no': 0, 'no2': 38.39, 'o3': 6...",1737507600,2025-01-22 01:00:00
2,{'aqi': 5},"{'co': 2109.53, 'no': 0.01, 'no2': 60.32, 'o3'...",1737511200,2025-01-22 02:00:00
3,{'aqi': 5},"{'co': 3631.59, 'no': 10.51, 'no2': 106.93, 'o...",1737514800,2025-01-22 03:00:00
4,{'aqi': 5},"{'co': 5821.23, 'no': 63.48, 'no2': 124.75, 'o...",1737518400,2025-01-22 04:00:00


In [27]:
df_main = raw_df['main'].apply(pd.Series)
df_components = raw_df['components'].apply(pd.Series)
df_components.head()

,co,no,no2,o3,so2,pm2_5,pm10,nh3
0,1201.63,0.00,28.45,72.24,10.73,66.30,85.46,8.04
1,1508.71,0.00,38.39,60.80,13.47,86.59,108.43,9.37
2,2109.53,0.01,60.32,41.48,17.40,122.00,148.70,13.05
3,3631.59,10.51,106.93,5.99,25.03,201.82,240.67,23.81
4,5821.23,63.48,124.75,6.62,35.29,299.41,356.46,36.98


In [29]:
df = pd.concat([raw_df.drop(['main', 'components'], axis=1), df_main, df_components], axis=1)
df.head()

,dt,datetime,aqi,co,no,no2,o3,so2,pm2_5,pm10,nh3
0,1737504000,2025-01-22 00:00:00,4.0,1201.63,0.00,28.45,72.24,10.73,66.30,85.46,8.04
1,1737507600,2025-01-22 01:00:00,5.0,1508.71,0.00,38.39,60.80,13.47,86.59,108.43,9.37
2,1737511200,2025-01-22 02:00:00,5.0,2109.53,0.01,60.32,41.48,17.40,122.00,148.70,13.05
3,1737514800,2025-01-22 03:00:00,5.0,3631.59,10.51,106.93,5.99,25.03,201.82,240.67,23.81
4,1737518400,2025-01-22 04:00:00,5.0,5821.23,63.48,124.75,6.62,35.29,299.41,356.46,36.98


In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8473 entries, 0 to 8472
Data columns (total 11 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   dt        8473 non-null   int64         
 1   datetime  8473 non-null   datetime64[ns]
 2   aqi       8473 non-null   float64       
 3   co        8473 non-null   float64       
 4   no        8473 non-null   float64       
 5   no2       8473 non-null   float64       
 6   o3        8473 non-null   float64       
 7   so2       8473 non-null   float64       
 8   pm2_5     8473 non-null   float64       
 9   pm10      8473 non-null   float64       
 10  nh3       8473 non-null   float64       
dtypes: datetime64[ns](1), float64(9), int64(1)
memory usage: 728.3 KB


In [31]:
df.sort_values('datetime', inplace=True)
df.reset_index(drop=True, inplace=True)
df.head()

,dt,datetime,aqi,co,no,no2,o3,so2,pm2_5,pm10,nh3
0,1737504000,2025-01-22 00:00:00,4.0,1201.63,0.00,28.45,72.24,10.73,66.30,85.46,8.04
1,1737507600,2025-01-22 01:00:00,5.0,1508.71,0.00,38.39,60.80,13.47,86.59,108.43,9.37
2,1737511200,2025-01-22 02:00:00,5.0,2109.53,0.01,60.32,41.48,17.40,122.00,148.70,13.05
3,1737514800,2025-01-22 03:00:00,5.0,3631.59,10.51,106.93,5.99,25.03,201.82,240.67,23.81
4,1737518400,2025-01-22 04:00:00,5.0,5821.23,63.48,124.75,6.62,35.29,299.41,356.46,36.98


In [35]:
df.to_csv("../Dataset/aqi_data.csv", index=False)